In [1]:
%%capture
!pip install timm transformers audiomentations --no-index --find-links=file:/kaggle/input/hms-pip-wheels3

In [2]:
import sys
sys.path.append(f'/kaggle/input/hms-3rd-place-weights/pytorch/src/2/configs/configs')
sys.path.append(f'/kaggle/input/hms-3rd-place-weights/pytorch/src/2/data/data')
sys.path.append(f'/kaggle/input/hms-3rd-place-weights/pytorch/src/2/models/models')
sys.path.append(f'/kaggle/input/hms-3rd-place-weights/pytorch/src/2/configs')

In [3]:
import numpy as np
import pandas as pd
import scipy as sp
import os
import json
import sys
import importlib
import multiprocessing as mp
import gc
from tqdm import tqdm
import glob
import torch
from copy import copy
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader
from sklearn.metrics import  mean_squared_error

torch.backends.cudnn.benchmark = True

In [4]:
COMP_FOLDER = '/kaggle/input/hms-harmful-brain-activity-classification/'

train_df = pd.read_csv(COMP_FOLDER + 'train.csv')
test_df = pd.read_csv(COMP_FOLDER + 'test.csv')
sample_submission = pd.read_csv(COMP_FOLDER + 'sample_submission.csv')

EEG_FOLDER = '/kaggle/input/hms-harmful-brain-activity-classification/test_eegs/'

PUBLIC_RUN = len(test_df) == 1
N_CORES = mp.cpu_count()
MIXED_PRECISION = False

RAM_CHECK = False
OOF_CHECK = False


DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if PUBLIC_RUN is False:
    RAM_CHECK = False
    OOF_CHECK = False

if OOF_CHECK is True:
    train_df = pd.read_csv('/kaggle/input/hms-aws-bucket/train_folded_17k.csv')
    EEG_FOLDER = '/kaggle/input/hms-harmful-brain-activity-classification/train_eegs/'
    test_df = train_df[train_df['fold']==0].copy()


if RAM_CHECK is True:
    train_df = pd.read_csv('/kaggle/input/hms-aws-bucket/train_folded_17k.csv')
    EEG_FOLDER = '/kaggle/input/hms-harmful-brain-activity-classification/train_eegs/'
    test_df = train_df[train_df['fold']==0].copy()
    test_df = test_df.head(2640).copy()

print(train_df.shape)
print(test_df.shape)

TARGETS = ['seizure_vote','lpd_vote','gpd_vote','lrda_vote','grda_vote','other_vote']
if TARGETS[0] not in test_df.columns:
    test_df[TARGETS] = 1
    test_df['eeg_label_offset_seconds'] = 0
    test_df['spectrogram_label_offset_seconds'] = 0

(106800, 15)
(1, 3)


In [5]:
def get_cfg(CFG):
    cfg = importlib.import_module('default_config')
    importlib.reload(cfg)
    cfg = importlib.import_module(CFG)
    importlib.reload(cfg)
    cfg = copy(cfg.cfg)
    cfg.data_dir = COMP_FOLDER
    cfg.mixed_precision = MIXED_PRECISION
    cfg.pretrained = False
    cfg.pretrained_weights = False
    cfg.offline_inference = True
    return cfg

def get_dl(cfg):
    ds = importlib.import_module(cfg.dataset)
    importlib.reload(ds)
    CustomDataset = ds.CustomDataset
    batch_to_device = ds.batch_to_device
    test_ds = CustomDataset(test_df, cfg, cfg.val_aug, mode="test")
    test_dl = DataLoader(test_ds, shuffle=False, batch_size=cfg.batch_size, collate_fn=ds.val_collate_fn, num_workers=N_CORES, pin_memory=True)
    return test_dl, batch_to_device

def get_state_dict(sd_fp):
    sd = torch.load(sd_fp, map_location="cpu")
    if "model" in sd.keys():
        sd = sd["model"]
    sd = {k.replace("module.", ""):v for k,v in sd.items()}
    return sd

def get_nets(cfg,state_dicts,test_ds):
    model = importlib.import_module(cfg.model)
    importlib.reload(model)
    Net = model.Net
    nets = []
    for i,state_dict in enumerate(state_dicts):
        net = Net(cfg).eval().to(DEVICE)
        print("loading dict")
        sd = get_state_dict(state_dict)
        net.load_state_dict(sd, strict=True)
        net.return_logits = True
        nets += [net]
        del sd
        gc.collect()
    return nets

In [6]:
def generate(name = 'cfg_1', weights_dir = './'):
    cfg = get_cfg(name)
    cfg.pretrained = False
    cfg.data_folder = EEG_FOLDER
    state_dict_fps = sorted(glob.glob(weights_dir, recursive = True))
    test_dl, batch_to_device = get_dl(cfg)
    print('\n'.join(state_dict_fps))
    nets = get_nets(cfg,state_dict_fps, test_dl.dataset)
    preds = []
    with torch.inference_mode():
        for batch in tqdm(test_dl):
            batch = batch_to_device(batch,DEVICE)
            outs = [net(batch) for net in nets]
            preds += [torch.stack([out['logits'] for out in outs], dim=0).mean(0).cpu()]
    preds = torch.cat(preds, dim=0).float()
    print('preds', preds.shape, ', test_df',test_df.shape)
    gc.collect()
    torch.cuda.empty_cache()
    return preds

In [7]:
PREDS = {}

In [8]:
PREDS['cfg_1'] = generate(name = 'cfg_1', weights_dir = f'/kaggle/input/hms-3rd-place-weights/pytorch/cfg_1/1/cfg_1/*/*check*')

mode:test, df.shape:(1, 11)
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_1/1/cfg_1/fold-1/checkpoint_last_seed130801.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_1/1/cfg_1/fold-1/checkpoint_last_seed279022.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_1/1/cfg_1/fold-1/checkpoint_last_seed357109.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_1/1/cfg_1/fold-1/checkpoint_last_seed368953.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_1/1/cfg_1/fold-1/checkpoint_last_seed500987.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_1/1/cfg_1/fold-1/checkpoint_last_seed632448.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_1/1/cfg_1/fold-1/checkpoint_last_seed781358.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_1/1/cfg_1/fold-1/checkpoint_last_seed784355.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_1/1/cfg_1/fold0/checkpoint_last_seed112466.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_1/1/cfg_1/fold0/checkpoint_last_seed46642.pth
/kaggle/input/h

100%|██████████| 1/1 [00:05<00:00,  5.29s/it]


preds torch.Size([1, 6]) , test_df (1, 11)


In [9]:
PREDS['cfg_2a'] = generate(name = 'cfg_2a', weights_dir = f'/kaggle/input/hms-3rd-place-weights/pytorch/cfg_2a/1/cfg_2a/*/*check*')

mode:test, df.shape:(1, 11)
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_2a/1/cfg_2a/fold-1/checkpoint_last_seed339403.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_2a/1/cfg_2a/fold-1/checkpoint_last_seed617434.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_2a/1/cfg_2a/fold-1/checkpoint_last_seed739903.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_2a/1/cfg_2a/fold-1/checkpoint_last_seed746094.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_2a/1/cfg_2a/fold-1/checkpoint_last_seed795609.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_2a/1/cfg_2a/fold-1/checkpoint_last_seed83140.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_2a/1/cfg_2a/fold-1/checkpoint_last_seed896170.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_2a/1/cfg_2a/fold-1/checkpoint_last_seed954117.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_2a/1/cfg_2a/fold0/checkpoint_last_seed468463.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_2a/1/cfg_2a/fold0/checkpoint_last_seed87487.

The cos_cached attribute will be removed in 4.39. Bear in mind that its contents changed in v4.38. Use the forward method of RoPE from now on instead. It is not used in the `LlamaAttention` class
The sin_cached attribute will be removed in 4.39. Bear in mind that its contents changed in v4.38. Use the forward method of RoPE from now on instead. It is not used in the `LlamaAttention` class


n_params: 717027
loading dict
n_params: 717027
loading dict
n_params: 717027
loading dict
n_params: 717027
loading dict
n_params: 717027
loading dict
n_params: 717027
loading dict
n_params: 717027
loading dict
n_params: 717027
loading dict
n_params: 717027
loading dict
n_params: 717027
loading dict
n_params: 717027
loading dict
n_params: 717027
loading dict
n_params: 717027
loading dict
n_params: 717027
loading dict
n_params: 717027
loading dict
n_params: 717027
loading dict


100%|██████████| 1/1 [00:01<00:00,  1.00s/it]


preds torch.Size([1, 6]) , test_df (1, 11)


In [10]:
PREDS['cfg_2b'] = generate(name = 'cfg_2b', weights_dir = f'/kaggle/input/hms-3rd-place-weights/pytorch/cfg_2b/1/cfg_2b/*/*check*')

mode:test, df.shape:(1, 11)
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_2b/1/cfg_2b/fold-1/checkpoint_last_seed212511.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_2b/1/cfg_2b/fold-1/checkpoint_last_seed236204.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_2b/1/cfg_2b/fold-1/checkpoint_last_seed267329.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_2b/1/cfg_2b/fold-1/checkpoint_last_seed420942.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_2b/1/cfg_2b/fold-1/checkpoint_last_seed708203.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_2b/1/cfg_2b/fold-1/checkpoint_last_seed744343.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_2b/1/cfg_2b/fold-1/checkpoint_last_seed871487.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_2b/1/cfg_2b/fold-1/checkpoint_last_seed98521.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_2b/1/cfg_2b/fold0/checkpoint_last_seed299313.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_2b/1/cfg_2b/fold0/checkpoint_last_seed495109

100%|██████████| 1/1 [00:00<00:00,  2.57it/s]


preds torch.Size([1, 6]) , test_df (1, 11)


In [11]:
PREDS['cfg_3'] = generate(name = 'cfg_3', weights_dir = f'/kaggle/input/hms-3rd-place-weights/pytorch/cfg_3/1/cfg_3/*/*check*')

mode:test, df.shape:(1, 11)
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_3/1/cfg_3/fold-1/checkpoint_last_seed115094.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_3/1/cfg_3/fold-1/checkpoint_last_seed122166.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_3/1/cfg_3/fold-1/checkpoint_last_seed138555.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_3/1/cfg_3/fold-1/checkpoint_last_seed23399.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_3/1/cfg_3/fold-1/checkpoint_last_seed346171.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_3/1/cfg_3/fold-1/checkpoint_last_seed44128.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_3/1/cfg_3/fold-1/checkpoint_last_seed478350.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_3/1/cfg_3/fold-1/checkpoint_last_seed52940.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_3/1/cfg_3/fold0/checkpoint_last_seed765856.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_3/1/cfg_3/fold0/checkpoint_last_seed8085.pth
/kaggle/input/hms-3

100%|██████████| 1/1 [00:01<00:00,  1.64s/it]


preds torch.Size([1, 6]) , test_df (1, 11)


In [12]:
PREDS['cfg_4'] = generate(name = 'cfg_4', weights_dir = f'/kaggle/input/hms-3rd-place-weights/pytorch/cfg_4/1/cfg_4/*/*check*')


mode:test, df.shape:(1, 11)
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_4/1/cfg_4/fold-1/checkpoint_last_seed133295.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_4/1/cfg_4/fold-1/checkpoint_last_seed160849.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_4/1/cfg_4/fold-1/checkpoint_last_seed172894.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_4/1/cfg_4/fold-1/checkpoint_last_seed387768.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_4/1/cfg_4/fold-1/checkpoint_last_seed474421.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_4/1/cfg_4/fold-1/checkpoint_last_seed514882.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_4/1/cfg_4/fold-1/checkpoint_last_seed517561.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_4/1/cfg_4/fold-1/checkpoint_last_seed545422.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_4/1/cfg_4/fold0/checkpoint_last_seed488325.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_4/1/cfg_4/fold0/checkpoint_last_seed585093.pth
/kaggle/input/

100%|██████████| 1/1 [00:02<00:00,  2.60s/it]


preds torch.Size([1, 6]) , test_df (1, 11)


In [13]:
PREDS['cfg_5a'] = generate(name = 'cfg_5a', weights_dir = f'/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5a/1/cfg_5a/*/*check*')

mode:test, df.shape:(1, 11)
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5a/1/cfg_5a/fold-1/checkpoint_last_seed250000.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5a/1/cfg_5a/fold-1/checkpoint_last_seed354931.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5a/1/cfg_5a/fold-1/checkpoint_last_seed423489.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5a/1/cfg_5a/fold-1/checkpoint_last_seed565488.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5a/1/cfg_5a/fold-1/checkpoint_last_seed725064.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5a/1/cfg_5a/fold-1/checkpoint_last_seed830974.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5a/1/cfg_5a/fold-1/checkpoint_last_seed854136.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5a/1/cfg_5a/fold-1/checkpoint_last_seed902559.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5a/1/cfg_5a/fold0/checkpoint_last_seed322323.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5a/1/cfg_5a/fold0/checkpoint_last_seed41813

100%|██████████| 1/1 [00:02<00:00,  2.54s/it]


preds torch.Size([1, 6]) , test_df (1, 11)


In [14]:
PREDS['cfg_5b'] = generate(name = 'cfg_5b', weights_dir = f'/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5b/1/cfg_5b/*/*check*')

mode:test, df.shape:(1, 11)
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5b/1/cfg_5b/fold-1/checkpoint_last_seed176925.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5b/1/cfg_5b/fold-1/checkpoint_last_seed303408.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5b/1/cfg_5b/fold-1/checkpoint_last_seed443624.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5b/1/cfg_5b/fold-1/checkpoint_last_seed612809.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5b/1/cfg_5b/fold-1/checkpoint_last_seed629103.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5b/1/cfg_5b/fold-1/checkpoint_last_seed921352.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5b/1/cfg_5b/fold-1/checkpoint_last_seed933678.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5b/1/cfg_5b/fold-1/checkpoint_last_seed94228.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5b/1/cfg_5b/fold0/checkpoint_last_seed395706.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5b/1/cfg_5b/fold0/checkpoint_last_seed931485

100%|██████████| 1/1 [00:00<00:00,  1.13it/s]


preds torch.Size([1, 6]) , test_df (1, 11)


In [15]:
PREDS['cfg_5c'] = generate(name = 'cfg_5c', weights_dir = f'/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5c/1/cfg_5c/*/*check*')

mode:test, df.shape:(1, 11)
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5c/1/cfg_5c/fold-1/checkpoint_last_seed357028.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5c/1/cfg_5c/fold-1/checkpoint_last_seed380199.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5c/1/cfg_5c/fold-1/checkpoint_last_seed40687.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5c/1/cfg_5c/fold-1/checkpoint_last_seed459284.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5c/1/cfg_5c/fold-1/checkpoint_last_seed514890.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5c/1/cfg_5c/fold-1/checkpoint_last_seed708319.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5c/1/cfg_5c/fold-1/checkpoint_last_seed770424.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5c/1/cfg_5c/fold-1/checkpoint_last_seed921744.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5c/1/cfg_5c/fold0/checkpoint_last_seed342524.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5c/1/cfg_5c/fold0/checkpoint_last_seed779073

100%|██████████| 1/1 [00:00<00:00,  1.12it/s]


preds torch.Size([1, 6]) , test_df (1, 11)


In [16]:
PREDS['cfg_5d'] = generate(name = 'cfg_5d', weights_dir = f'/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5d/1/cfg_5d/*/*check*')

mode:test, df.shape:(1, 11)
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5d/1/cfg_5d/fold-1/checkpoint_last_seed215014.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5d/1/cfg_5d/fold-1/checkpoint_last_seed398725.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5d/1/cfg_5d/fold-1/checkpoint_last_seed429062.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5d/1/cfg_5d/fold-1/checkpoint_last_seed482501.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5d/1/cfg_5d/fold-1/checkpoint_last_seed688240.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5d/1/cfg_5d/fold-1/checkpoint_last_seed842310.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5d/1/cfg_5d/fold-1/checkpoint_last_seed907596.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5d/1/cfg_5d/fold0/checkpoint_last_seed791089.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5d/1/cfg_5d/fold0/checkpoint_last_seed819068.pth
/kaggle/input/hms-3rd-place-weights/pytorch/cfg_5d/1/cfg_5d/fold1/checkpoint_last_seed215120

100%|██████████| 1/1 [00:04<00:00,  4.07s/it]


preds torch.Size([1, 6]) , test_df (1, 11)


In [17]:
CLASS_BIAS = [ 0.012535,  0.03458 ,  0.01761 ,  0.05957 , -0.02608 , -0.0982  ]

In [18]:
WEIGHTS = {'cfg_1': 0.12932,
 'cfg_2a': 0.14089,
 'cfg_2b': 0.12269,
 'cfg_3': 0.1174,
 'cfg_4': 0.10538,
 'cfg_5a': 0.0987,
 'cfg_5b': 0.15285,
 'cfg_5c': 0.10973,
 'cfg_5d': 0.09444}

In [19]:
assert sorted(PREDS.keys()) == sorted(WEIGHTS.keys())


In [20]:
for k,p in PREDS.items():
    print(k)
    print(np.around(p[:3].softmax(1).numpy(), decimals=3))


cfg_1
[[0.008 0.025 0.001 0.388 0.058 0.519]]
cfg_2a
[[0.022 0.007 0.001 0.332 0.017 0.621]]
cfg_2b
[[0.021 0.008 0.001 0.371 0.015 0.585]]
cfg_3
[[0.016 0.031 0.    0.433 0.021 0.498]]
cfg_4
[[0.007 0.023 0.    0.417 0.023 0.529]]
cfg_5a
[[0.005 0.024 0.    0.456 0.026 0.489]]
cfg_5b
[[0.004 0.03  0.001 0.426 0.035 0.504]]
cfg_5c
[[0.003 0.019 0.    0.428 0.032 0.518]]
cfg_5d
[[0.006 0.038 0.    0.52  0.035 0.4  ]]


In [21]:
total_weights = sum(list(WEIGHTS.values()))
total_weights

1.0714000000000001

In [22]:
WEIGHTS = {k:v/sum(list(WEIGHTS.values())) for k,v in WEIGHTS.items()}
sum(list(WEIGHTS.values()))

0.9999999999999999

In [23]:
for k,v in WEIGHTS.items():
    print(f'{v:0.3f} {k}')

0.121 cfg_1
0.132 cfg_2a
0.115 cfg_2b
0.110 cfg_3
0.098 cfg_4
0.092 cfg_5a
0.143 cfg_5b
0.102 cfg_5c
0.088 cfg_5d


In [24]:
preds = torch.stack([(v * WEIGHTS[k]) for k,v in PREDS.items()]).sum(0)


In [25]:
print(np.around(preds[:3].numpy(), decimals=3))


[[-1.636 -0.812 -4.638  2.259 -0.462  2.494]]


In [26]:
postproc = torch.tensor(CLASS_BIAS).unsqueeze(0)


In [27]:
preds = preds + postproc


In [28]:
print(np.around(preds[:3].numpy(), decimals=3))


[[-1.623 -0.777 -4.62   2.319 -0.489  2.396]]


In [29]:
preds = preds.softmax(1).numpy().copy()


In [30]:
preds = preds / preds.sum(1)[:,None]


In [31]:
print(np.around(preds[:3], decimals=3))


[[0.009 0.02  0.    0.453 0.027 0.49 ]]


In [32]:
sub = pd.DataFrame({'eeg_id': test_df.eeg_id.values})
sub[TARGETS] = preds
sub.to_csv('submission.csv',index=False)
print('Submission shape',sub.shape)
sub.head()

Submission shape (1, 7)


,eeg_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,3911565283,0.008796,0.020498,0.000439,0.453273,0.027365,0.489628
